# Baseline Models — Predicting Smartphone Addiction

This notebook establishes reproducible machine-learning baselines for the Kaggle Playground Series S6E8 binary classification task.

The purpose is not to maximize leaderboard score immediately, but to answer a more useful modeling question:

> How much predictive performance can we obtain from standard linear and gradient-boosted models before advanced feature engineering?

**Metric:** ROC-AUC  
**Validation:** 5-fold stratified cross-validation with a fixed random seed


## 1. Imports and data loading

The notebook supports both the Kaggle environment and the local repository layout.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

kaggle_dir = Path("/kaggle/input/competitions/playground-series-s6e8")
local_dir = Path("../data")

data_dir = kaggle_dir if kaggle_dir.exists() else local_dir

train = pd.read_csv(data_dir / "train.csv")
test = pd.read_csv(data_dir / "test.csv")

X = train.drop(columns=["id", "addicted_label"])
y = train["addicted_label"]

print("Data directory:", data_dir)
print("X shape:", X.shape)
print("Target positive rate:", f"{y.mean():.4f}")


## 2. Validation strategy

All models use the same 5-fold `StratifiedKFold` split:

- `n_splits=5`
- `shuffle=True`
- `random_state=42`

Using identical folds makes model comparisons fair and preserves the approximately 71/29 class ratio in every fold.


In [7]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

## 3. Feature groups and preprocessing

The dataset contains 9 numerical and 3 categorical predictors.

For the linear baseline:
- numerical missing values are median-imputed and standardized,
- categorical missing values are mode-imputed and one-hot encoded.

For tree models, scaling is unnecessary.


In [8]:
numerical_cols = [
    "age",
    "daily_screen_time_hours",
    "social_media_hours",
    "gaming_hours",
    "work_study_hours",
    "sleep_hours",
    "notifications_per_day",
    "app_opens_per_day",
    "weekend_screen_time"
]

categorical_cols = [
    "gender",
    "stress_level",
    "academic_work_impact"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

## 4. Logistic Regression baseline

Logistic Regression provides a useful low-complexity reference. If a nonlinear model cannot meaningfully outperform it, added complexity would be difficult to justify.


In [9]:
logistic_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=500))
])

In [10]:
logistic_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):

    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    logistic_model.fit(
        X_train_fold,
        y_train_fold
    )

    val_probabilities = logistic_model.predict_proba(
        X_val_fold
    )[:, 1]

    auc = roc_auc_score(
        y_val_fold,
        val_probabilities
    )

    logistic_scores.append(auc)

    print(f"Fold {fold} AUC: {auc:.6f}")

print("-" * 40)
print(f"Mean CV AUC: {np.mean(logistic_scores):.6f}")
print(f"Std CV AUC:  {np.std(logistic_scores):.6f}")

Fold 1 AUC: 0.910346
Fold 2 AUC: 0.910797
Fold 3 AUC: 0.911877
Fold 4 AUC: 0.912653
Fold 5 AUC: 0.911573
----------------------------------------
Mean CV AUC: 0.911449
Std CV AUC:  0.000811


**5-fold mean ROC-AUC: 0.911449 ± 0.000811**

The fold-to-fold variation is small, so the estimate is stable.  
However, the score leaves substantial room for nonlinear models, which is consistent with the strongly nonlinear patterns observed during EDA.


## 5. LightGBM baseline

LightGBM is the first gradient-boosted tree baseline. Numerical values are median-imputed, while categorical variables are mode-imputed and one-hot encoded.


In [12]:
from lightgbm import LGBMClassifier
tree_preprocessor = ColumnTransformer([
    (
        "num",
        SimpleImputer(strategy="median"),
        numerical_cols
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_cols
    )
])

lightgbm_model = Pipeline([
    ("preprocessing", tree_preprocessor),
    (
        "model",
        LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        )
    )
])

In [13]:
lightgbm_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):

    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    lightgbm_model.fit(
        X_train_fold,
        y_train_fold
    )

    val_probabilities = lightgbm_model.predict_proba(
        X_val_fold
    )[:, 1]

    auc = roc_auc_score(
        y_val_fold,
        val_probabilities
    )

    lightgbm_scores.append(auc)

    print(f"Fold {fold} AUC: {auc:.6f}")

print("-" * 40)
print(f"Mean CV AUC: {np.mean(lightgbm_scores):.6f}")
print(f"Std CV AUC:  {np.std(lightgbm_scores):.6f}")

Fold 1 AUC: 0.959176
Fold 2 AUC: 0.959827
Fold 3 AUC: 0.960412
Fold 4 AUC: 0.960928
Fold 5 AUC: 0.959771
----------------------------------------
Mean CV AUC: 0.960023
Std CV AUC:  0.000598


**5-fold mean ROC-AUC: 0.960023 ± 0.000598**

This is a large improvement over Logistic Regression:

**0.91145 → 0.96002**

The result confirms that nonlinear interactions and threshold-like behavior are important in this dataset.


## 6. CatBoost baseline

CatBoost is tested because it can work directly with categorical features and can natively handle missing numerical values.

Categorical missing values are represented explicitly as `"Missing"`.


In [16]:
from catboost import CatBoostClassifier
X_catboost = X.copy()

# CatBoost can handle missing numerical values natively,
# but categorical missing values should be represented as a category.
X_catboost[categorical_cols] = (
    X_catboost[categorical_cols]
    .fillna("Missing")
)

catboost_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=7,
    eval_metric="AUC",
    random_seed=42,
    verbose=0,
    allow_writing_files=False
)

In [17]:
catboost_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_catboost, y), start=1):

    X_train_fold = X_catboost.iloc[train_idx]
    X_val_fold = X_catboost.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    catboost_model.fit(
        X_train_fold,
        y_train_fold,
        cat_features=categorical_cols
    )

    val_probabilities = catboost_model.predict_proba(
        X_val_fold
    )[:, 1]

    auc = roc_auc_score(
        y_val_fold,
        val_probabilities
    )

    catboost_scores.append(auc)

    print(f"Fold {fold} AUC: {auc:.6f}")

print("-" * 40)
print(f"Mean CV AUC: {np.mean(catboost_scores):.6f}")
print(f"Std CV AUC:  {np.std(catboost_scores):.6f}")

Fold 1 AUC: 0.952620
Fold 2 AUC: 0.953163
Fold 3 AUC: 0.953738
Fold 4 AUC: 0.954481
Fold 5 AUC: 0.953299
----------------------------------------
Mean CV AUC: 0.953460
Std CV AUC:  0.000623


**5-fold mean ROC-AUC: 0.953460 ± 0.000623**

CatBoost clearly outperforms the linear baseline, but in this setup LightGBM performs better.


## 7. Baseline comparison


In [18]:
baseline_results = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "LightGBM",
        "CatBoost"
    ],
    "mean_cv_auc": [
        np.mean(logistic_scores),
        np.mean(lightgbm_scores),
        np.mean(catboost_scores)
    ],
    "std_cv_auc": [
        np.std(logistic_scores),
        np.std(lightgbm_scores),
        np.std(catboost_scores)
    ]
})

baseline_results.sort_values(
    "mean_cv_auc",
    ascending=False
)

,model,mean_cv_auc,std_cv_auc
1,LightGBM,0.960023,0.000598
2,CatBoost,0.953460,0.000623
0,Logistic Regression,0.911449,0.000811


In [ ]:
import matplotlib.pyplot as plt

results_plot = baseline_results.sort_values("mean_cv_auc")

plt.figure(figsize=(7, 4))
plt.barh(results_plot["model"], results_plot["mean_cv_auc"])
plt.xlabel("Mean 5-fold ROC-AUC")
plt.title("Baseline Model Comparison")
plt.xlim(0.90, 0.965)

for i, value in enumerate(results_plot["mean_cv_auc"]):
    plt.text(value + 0.0005, i, f"{value:.5f}", va="center")

plt.tight_layout()
plt.show()


| Model | Mean CV ROC-AUC | Std |
|---|---:|---:|
| Logistic Regression | 0.911449 | 0.000811 |
| CatBoost | 0.953460 | 0.000623 |
| **LightGBM** | **0.960023** | **0.000598** |

### Interpretation

Three conclusions matter for the next stage:

1. **Nonlinearity matters.** Both boosting models substantially outperform Logistic Regression.
2. **Validation is stable.** Standard deviations are around \(6\times10^{-4}\) to \(8\times10^{-4}\), so small changes must be judged carefully.
3. **The baseline ceiling is already high.** Future improvements should come from better representations, stronger tree configurations, or leakage-safe feature engineering rather than simply adding model complexity.

The next notebook, `03_xgboost_advanced.ipynb`, builds on this baseline and reaches approximately **0.96774 CV ROC-AUC** with independently developed structural features and nested cross-fitted target encoding.
